# Schema Drift Detection Helpers

This notebook provides schema drift detection, alerting, and monitoring capabilities.

## Features
- Compare actual schema against expected contract
- Detect added, removed, and modified columns
- Log drift events to monitoring table
- Alert on critical schema changes
- Support for both blocking and non-blocking drift handling

In [ ]:
from pyspark.sql.types import StructType, StructField, DataType, ArrayType, StringType, BooleanType, TimestampType
from datetime import datetime
import json
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# ============================================================================
# SCHEMA COMPARISON UTILITIES
# ============================================================================

def compare_schemas(expected: StructType, actual: StructType) -> dict:
    """
    Compare two schemas and return detailed drift information.
    
    Args:
        expected: Expected schema (contract)
        actual: Actual schema from data source
    
    Returns:
        Dictionary with drift details:
        - has_drift: bool
        - added_columns: list of new columns
        - removed_columns: list of missing columns
        - type_changes: list of columns with type changes
        - severity: 'NONE' | 'WARNING' | 'CRITICAL'
    """
    expected_fields = {f.name: f for f in expected.fields}
    actual_fields = {f.name: f for f in actual.fields}
    
    expected_names = set(expected_fields.keys())
    actual_names = set(actual_fields.keys())
    
    added_columns = list(actual_names - expected_names)
    removed_columns = list(expected_names - actual_names)
    
    # Check for type changes in common columns
    type_changes = []
    common_columns = expected_names & actual_names
    for col_name in common_columns:
        expected_type = str(expected_fields[col_name].dataType)
        actual_type = str(actual_fields[col_name].dataType)
        if expected_type != actual_type:
            type_changes.append({
                "column": col_name,
                "expected_type": expected_type,
                "actual_type": actual_type
            })
    
    has_drift = bool(added_columns or removed_columns or type_changes)
    
    # Determine severity
    if not has_drift:
        severity = "NONE"
    elif removed_columns or type_changes:
        severity = "CRITICAL"  # Breaking changes
    else:
        severity = "WARNING"  # Only additive changes
    
    return {
        "has_drift": has_drift,
        "added_columns": added_columns,
        "removed_columns": removed_columns,
        "type_changes": type_changes,
        "severity": severity,
        "expected_column_count": len(expected_fields),
        "actual_column_count": len(actual_fields),
        "detected_at": datetime.now().isoformat()
    }


def schema_to_dict(schema: StructType) -> dict:
    """
    Convert Spark schema to dictionary for storage/comparison.
    """
    return {
        "columns": [
            {
                "name": field.name,
                "type": str(field.dataType),
                "nullable": field.nullable
            }
            for field in schema.fields
        ],
        "column_count": len(schema.fields)
    }


def dict_to_schema(schema_dict: dict) -> StructType:
    """
    Convert dictionary back to Spark schema.
    Note: This is simplified and may not handle complex nested types.
    """
    from pyspark.sql.types import (
        IntegerType, StringType, LongType, DoubleType, 
        TimestampType, BooleanType, DecimalType, StructType, StructField
    )
    
    type_mapping = {
        "IntegerType": IntegerType(),
        "Integer": IntegerType(),
        "StringType": StringType(),
        "String": StringType(),
        "LongType": LongType(),
        "Long": LongType(),
        "DoubleType": DoubleType(),
        "Double": DoubleType(),
        "TimestampType": TimestampType(),
        "Timestamp": TimestampType(),
        "BooleanType": BooleanType(),
        "Boolean": BooleanType(),
    }
    
    fields = []
    for col in schema_dict.get("columns", []):
        col_type = type_mapping.get(col["type"], StringType())
        fields.append(StructField(col["name"], col_type, col.get("nullable", True)))
    
    return StructType(fields)

In [ ]:
# ============================================================================
# SCHEMA DRIFT MONITORING TABLE
# ============================================================================

def create_schema_drift_table(spark, catalog: str, schema: str) -> None:
    """
    Create the schema drift monitoring table.
    
    Args:
        spark: SparkSession
        catalog: Unity Catalog name
        schema: Schema name for monitoring table
    """
    table_fqn = f"{catalog}.{schema}.schema_drift_log"
    
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_fqn} (
            drift_id STRING,
            table_name STRING,
            source_system STRING,
            severity STRING,
            added_columns ARRAY<STRING>,
            removed_columns ARRAY<STRING>,
            type_changes STRING,
            expected_schema STRING,
            actual_schema STRING,
            detected_at TIMESTAMP,
            acknowledged BOOLEAN,
            acknowledged_by STRING,
            acknowledged_at TIMESTAMP,
            resolution_notes STRING,
            pipeline_run_id STRING
        ) USING DELTA
        TBLPROPERTIES (
            'delta.logRetentionDuration' = 'interval 365 days'
        )
    """)
    logger.info(f"Schema drift table created: {table_fqn}")


def log_schema_drift(
    spark,
    drift_table: str,
    table_name: str,
    drift_result: dict,
    source_system: str = "unknown",
    pipeline_run_id: str = None,
    expected_schema: StructType = None,
    actual_schema: StructType = None
) -> str:
    """
    Log schema drift event to monitoring table.
    
    Args:
        spark: SparkSession
        drift_table: Fully qualified drift log table name
        table_name: Name of table with drift
        drift_result: Result from compare_schemas()
        source_system: Source system identifier
        pipeline_run_id: Optional pipeline run identifier
        expected_schema: Optional expected schema for logging
        actual_schema: Optional actual schema for logging
    
    Returns:
        drift_id: Unique identifier for this drift event
    """
    import uuid
    
    drift_id = str(uuid.uuid4())
    
    drift_record_schema = StructType([
        StructField("drift_id", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("source_system", StringType(), False),
        StructField("severity", StringType(), False),
        StructField("added_columns", ArrayType(StringType()), False),
        StructField("removed_columns", ArrayType(StringType()), False),
        StructField("type_changes", StringType(), False),
        StructField("expected_schema", StringType(), True),
        StructField("actual_schema", StringType(), True),
        StructField("detected_at", TimestampType(), False),
        StructField("acknowledged", BooleanType(), False),
        StructField("acknowledged_by", StringType(), True),
        StructField("acknowledged_at", TimestampType(), True),
        StructField("resolution_notes", StringType(), True),
        StructField("pipeline_run_id", StringType(), True)
    ])
    
    drift_record = spark.createDataFrame([(
        drift_id,
        table_name,
        source_system,
        drift_result["severity"],
        drift_result["added_columns"],
        drift_result["removed_columns"],
        json.dumps(drift_result["type_changes"]),
        json.dumps(schema_to_dict(expected_schema)) if expected_schema else None,
        json.dumps(schema_to_dict(actual_schema)) if actual_schema else None,
        datetime.fromisoformat(drift_result["detected_at"]),
        False,
        None,
        None,
        None,
        pipeline_run_id
    )], schema=drift_record_schema)
    
    drift_record.write.format("delta").mode("append").saveAsTable(drift_table)
    
    logger.info(f"Logged schema drift event: {drift_id} for table {table_name}")
    return drift_id

In [ ]:
# ============================================================================
# ALERTING MECHANISMS
# ============================================================================

class SchemaDriftException(Exception):
    """Exception raised when critical schema drift is detected."""
    pass


def send_alert(
    drift_result: dict,
    table_name: str,
    alert_channel: str = "log",
    webhook_url: str = None,
    email_recipients: list = None
) -> None:
    """
    Send alert for schema drift based on configured channel.
    
    Args:
        drift_result: Result from compare_schemas()
        table_name: Table with schema drift
        alert_channel: One of 'log', 'webhook', 'email', 'all'
        webhook_url: Optional webhook URL for Slack/Teams
        email_recipients: Optional list of email addresses
    """
    severity = drift_result["severity"]
    
    # Build alert message
    alert_msg = f"🚨 SCHEMA DRIFT DETECTED - {severity}\n"
    alert_msg += f"Table: {table_name}\n"
    alert_msg += f"Detected at: {drift_result['detected_at']}\n"
    
    if drift_result["added_columns"]:
        alert_msg += f"➕ Added columns: {', '.join(drift_result['added_columns'])}\n"
    
    if drift_result["removed_columns"]:
        alert_msg += f"➖ Removed columns: {', '.join(drift_result['removed_columns'])}\n"
    
    if drift_result["type_changes"]:
        for tc in drift_result["type_changes"]:
            alert_msg += f"🔄 Type change: {tc['column']} ({tc['expected_type']} → {tc['actual_type']})\n"
    
    if severity == "CRITICAL":
        alert_msg += "\n⚠️ ACTION REQUIRED: Critical schema changes detected. Pipeline may fail."
    
    # Log alert
    if alert_channel in ["log", "all"]:
        if severity == "CRITICAL":
            logger.error(alert_msg)
        else:
            logger.warning(alert_msg)
    
    # Send webhook (Slack/Teams)
    if alert_channel in ["webhook", "all"] and webhook_url:
        _send_webhook_alert(webhook_url, table_name, drift_result, alert_msg)
    
    # Send email (requires SMTP configuration)
    if alert_channel in ["email", "all"] and email_recipients:
        _send_email_alert(email_recipients, table_name, drift_result, alert_msg)


def _send_webhook_alert(webhook_url: str, table_name: str, drift_result: dict, message: str) -> None:
    """
    Send alert to Slack or Microsoft Teams webhook.
    """
    import urllib.request
    import json
    
    # Slack format
    payload = {
        "text": "Schema Drift Alert",
        "attachments": [
            {
                "color": "danger" if drift_result["severity"] == "CRITICAL" else "warning",
                "fields": [
                    {"title": "Table", "value": table_name, "short": True},
                    {"title": "Severity", "value": drift_result["severity"], "short": True},
                    {"title": "Details", "value": message, "short": False}
                ]
            }
        ]
    }
    
    try:
        data = json.dumps(payload).encode('utf-8')
        req = urllib.request.Request(
            webhook_url,
            data=data,
            headers={'Content-Type': 'application/json'}
        )
        urllib.request.urlopen(req)
        logger.info("Webhook alert sent successfully")
    except Exception as e:
        logger.error(f"Failed to send webhook alert: {e}")


def _send_email_alert(recipients: list, table_name: str, drift_result: dict, message: str) -> None:
    """
    Send email alert (placeholder - requires SMTP configuration).
    """
    logger.info(f"Email alert would be sent to {recipients} for table {table_name}")
    # Implementation depends on your email service (SMTP, SendGrid, SES, etc.)

In [ ]:
# ============================================================================
# SCHEMA VALIDATION DECISION ENGINE
# ============================================================================

def validate_schema_with_policy(
    spark,
    expected_schema: StructType,
    actual_schema: StructType,
    table_name: str,
    drift_table: str = None,
    source_system: str = "unknown",
    pipeline_run_id: str = None,
    policy: str = "strict",
    alert_channel: str = "log",
    webhook_url: str = None,
    allow_additive_only: bool = True
) -> tuple[bool, dict]:
    """
    Validate schema against contract with configurable policy.
    
    Args:
        spark: SparkSession
        expected_schema: Expected schema contract
        actual_schema: Actual schema from source
        table_name: Name of table being validated
        drift_table: Fully qualified drift log table name
        source_system: Source system identifier
        pipeline_run_id: Optional pipeline run identifier
        policy: One of:
            - 'strict': Block on any drift
            - 'additive_only': Allow new columns, block removals/type changes
            - 'permissive': Log drift but never block
        alert_channel: Alert delivery channel
        webhook_url: Optional webhook URL
        allow_additive_only: If True, allow additive changes even in strict mode
    
    Returns:
        Tuple of (is_valid: bool, drift_result: dict)
    """
    # Compare schemas
    drift_result = compare_schemas(expected_schema, actual_schema)
    
    # No drift - all good
    if not drift_result["has_drift"]:
        return True, drift_result
    
    # Log drift event
    if drift_table:
        log_schema_drift(
            spark, drift_table, table_name, drift_result,
            source_system, pipeline_run_id, expected_schema, actual_schema
        )
    
    # Send alert
    send_alert(drift_result, table_name, alert_channel, webhook_url)
    
    # Apply policy
    if policy == "permissive":
        # Log but don't block
        return True, drift_result
    
    elif policy == "additive_only":
        # Block on removals or type changes, allow additions
        if drift_result["removed_columns"] or drift_result["type_changes"]:
            raise SchemaDriftException(
                f"Breaking schema change detected in {table_name}: "
                f"removed={drift_result['removed_columns']}, "
                f"type_changes={drift_result['type_changes']}"
            )
        # Additive only - allow but warn
        return True, drift_result
    
    elif policy == "strict":
        # Block on any drift
        if allow_additive_only and not drift_result["removed_columns"] and not drift_result["type_changes"]:
            # Additive changes allowed
            return True, drift_result
        raise SchemaDriftException(
            f"Schema drift blocked pipeline for {table_name}. "
            f"Severity: {drift_result['severity']}. "
            f"Added: {drift_result['added_columns']}, "
            f"Removed: {drift_result['removed_columns']}, "
            f"Type changes: {drift_result['type_changes']}"
        )
    
    # Default: block on critical, warn on warning
    if drift_result["severity"] == "CRITICAL":
        raise SchemaDriftException(f"Critical schema drift in {table_name}")
    
    return True, drift_result

## Usage Example

```python
# Run this helper notebook
%run ./helpers/NB_schema_drift_helpers

# Define expected schema contract
from pyspark.sql.types import *
expected = StructType([
    StructField("id", IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("price", DecimalType(12,2))
])

# Get actual schema from Bronze table
actual_df = spark.table("bronze.orders")
actual = actual_df.schema

# Validate with policy
is_valid, drift = validate_schema_with_policy(
    spark,
    expected_schema=expected,
    actual_schema=actual,
    table_name="bronze.orders",
    drift_table="workspace.monitoring.schema_drift_log",
    source_system="postgres_cdc",
    policy="additive_only",
    alert_channel="webhook",
    webhook_url="https://hooks.slack.com/..."
)

if is_valid:
    print("Schema validation passed")
else:
    print("Schema validation failed - pipeline blocked")
```